In [ ]:
#!pip install dash

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.2/7.2 MB 5.9 MB/s eta 0:00:02
   --- ------------------------------------ 0.6/7.2 MB 7.5 MB/s eta 0:00:01
   ------ --------------------------------- 1.1/7.2 MB 8.7 MB/s eta 0:00:01
   --------- ------------------------------ 1.6/7.2 MB 10.4 MB/s eta 0:00:01
   ----------- ---------------------------- 2.1/7.2 MB 10.2 MB/s eta 0:00:01
   --------------- ------------------------ 2.7/7.2 MB 10.9 MB/s eta 0:00:01
   ------------------ --------------------- 3.3/7.2 MB 11.0 MB/s eta 0:00:01
   --------------------- ------------------ 3.8/7.2 MB 11.0 MB/s eta 0:00:01
   ---------------------- ----------------- 4.1/7.2 MB 10.6 MB/s eta 0:00:01
   ------------------------- -------------- 4.6/7.2 MB 10.6 MB/s eta 0:00:01
   ---------------------------- ----------- 5.2/7.2 MB 10.7 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# === Core ===
import numpy as np
import pandas as pd

# === Visualization ===
import plotly.express as px
import plotly.graph_objects as go

# === Dash ===
from dash import Dash, dcc, html, Input, Output

# === Utilities ===
from datetime import datetime, timedelta
import random

In [4]:
# === Synthetic Data Generator ===

def generate_car_data(car_id, country, city, duration_sec=300, freq=10):
    n = duration_sec * freq
    t = np.linspace(0, duration_sec, n)

    base_hr = np.random.uniform(60, 85)

    # Heart rate variability
    hr = base_hr + 5 * np.sin(0.1 * t)

    # Noise
    noise = np.random.normal(0, 2, n)

    # Motion artifacts
    motion = np.random.rand(n) < 0.01
    hr[motion] += np.random.normal(20, 5, motion.sum())

    # Dropouts
    dropout = np.random.rand(n) < 0.005
    hr[dropout] = np.nan

    # Signal quality (derived, not random)
    quality = np.clip(1 - (np.abs(noise)/5 + motion*0.5 + dropout*0.7), 0, 1)

    timestamps = [datetime.now() + timedelta(seconds=i/freq) for i in range(n)]

    return pd.DataFrame({
        "timestamp": timestamps,
        "car_id": car_id,
        "country": country,
        "city": city,
        "heart_rate": hr + noise,
        "signal_quality": quality,
        "motion_artifact": motion,
        "dropout": dropout
    })

In [5]:
# === Generate Data for 3 Cars ===

cars = [
    ("car_1", "Sweden", "Gothenburg"),
    ("car_2", "Germany", "Munich"),
    ("car_3", "USA", "San Francisco")
]

df_list = [generate_car_data(*car) for car in cars]
df = pd.concat(df_list).reset_index(drop=True)

df.head()

,timestamp,car_id,country,city,heart_rate,signal_quality,motion_artifact,dropout
0,2026-04-21 21:31:26.346499,car_1,Sweden,Gothenburg,78.170944,0.275374,False,False
1,2026-04-21 21:31:26.446499,car_1,Sweden,Gothenburg,85.280479,0.312723,False,False
2,2026-04-21 21:31:26.546499,car_1,Sweden,Gothenburg,101.121685,0.494488,True,False
3,2026-04-21 21:31:26.646499,car_1,Sweden,Gothenburg,82.140746,0.960672,False,False
4,2026-04-21 21:31:26.746499,car_1,Sweden,Gothenburg,80.598791,0.720940,False,False


In [6]:
# === Save dataset ===
df.to_csv("../data/synthetic_data.csv", index=False)

In [7]:
# === Reliability Metric ===

def compute_reliability(df):
    df = df.copy()

    # Smoothness (penalize large jumps)
    df["hr_diff"] = df.groupby("car_id")["heart_rate"].diff().abs()
    smoothness = np.exp(-df["hr_diff"].fillna(0) / 10)

    df["reliability"] = (
        0.5 * df["signal_quality"] +
        0.3 * smoothness +
        0.2 * (~df["dropout"]).astype(int)
    )

    return df

df = compute_reliability(df)

In [8]:
# === Minimal Dash App ===

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Radar Dashboard MVP"),

    dcc.Dropdown(
        id="car-dropdown",
        options=[{"label": c, "value": c} for c in df["car_id"].unique()],
        value="car_1"
    ),

    dcc.Graph(id="hr-graph"),
    dcc.Graph(id="reliability-graph")
])

In [9]:
@app.callback(
    Output("hr-graph", "figure"),
    Output("reliability-graph", "figure"),
    Input("car-dropdown", "value")
)
def update_graphs(selected_car):

    dff = df[df["car_id"] == selected_car]

    fig_hr = px.line(dff, x="timestamp", y="heart_rate",
                     title="Heart Rate")

    fig_rel = px.line(dff, x="timestamp", y="reliability",
                      title="Reliability")

    return fig_hr, fig_rel

In [10]:
app.run(debug=True)